# Parallax 3D GIF Generator

Turn any single image into a 3D parallax GIF using monocular depth estimation, subject segmentation, and LaMa inpainting.

**Pipeline:**
1. Depth estimation (Depth Anything V2 **Large** — 335M params)
2. Subject segmentation (RMBG-2.0) with soft alpha + edge-aware matting
3. Multi-layer depth inpainting (LaMa) with depth-adaptive DoF blur
4. Sub-pixel orbital rendering (`cv2.remap` with Lanczos4 interpolation + sinusoidal easing)
5. GIF + MP4 assembly with optimized quantization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanpd13/parallax-3d-gif/blob/main/parallax_3d_gif.ipynb)

## 1. Setup

Install dependencies and upload your image.

In [ ]:
import subprocess, sys

def _is_installed(pkg):
    try:
        __import__(pkg)
        return True
    except ImportError:
        return False

if not _is_installed('kornia') or not _is_installed('simple_lama_inpainting'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'torch', 'torchvision', 'opencv-python-headless', 'Pillow', 'scipy',
        'tqdm', 'huggingface_hub', 'transformers>=4.45,<5.0',
        'simple-lama-inpainting', 'kornia'])
else:
    print('Dependencies already installed')

In [ ]:
import os

# HF login for gated models (RMBG-2.0 requires it)
# Pass token via: HF_TOKEN env var, Colab secrets, or interactive login
_hf_token = os.environ.get('HF_TOKEN')

if not _hf_token:
    try:
        from google.colab import userdata
        _hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass

if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token)
    print('Logged in to HF with token')
elif 'COLAB_RELEASE_TAG' in os.environ:
    from huggingface_hub import login
    login()  # interactive prompt
else:
    # Try to use cached credentials
    from huggingface_hub import HfFolder
    if HfFolder.get_token():
        print('Using cached HF credentials')
    else:
        print('WARNING: No HF token found. Gated models (RMBG-2.0) will fail.')

In [ ]:
import glob

# If INPUT is already defined (e.g. by papermill or a preceding cell), use it.
# Otherwise, look for any image in the current directory, or prompt Colab upload.
if 'INPUT' not in dir() or not os.path.isfile(INPUT):
    # Check for existing images in cwd
    existing = glob.glob('*.jpg') + glob.glob('*.jpeg') + glob.glob('*.png') + glob.glob('*.webp')
    # Exclude pipeline outputs
    existing = [f for f in existing if f not in ('depth.png', 'mask.png', 'bg_inpainted.png')]

    if existing:
        INPUT = existing[0]
        print(f'Found image: {INPUT}')
    elif 'COLAB_RELEASE_TAG' in os.environ:
        from google.colab import files
        uploaded = files.upload()
        INPUT = list(uploaded.keys())[0]
    else:
        raise FileNotFoundError('No input image found. Place a .jpg/.png in the working directory or set INPUT.')

print(f'Ready: {INPUT}')

## 2. Configuration

Adjust these parameters to control the 3D effect.

In [ ]:
from pathlib import Path

# -- Animation --
FRAMES        = FRAMES        if 'FRAMES'    in dir() else 9
ARC_DEG       = ARC_DEG       if 'ARC_DEG'   in dir() else 7.0
FPS           = FPS           if 'FPS'        in dir() else 12
PING_PONG     = PING_PONG     if 'PING_PONG' in dir() else True

# -- Image --
MAX_DIM       = MAX_DIM       if 'MAX_DIM'   in dir() else 2048

# -- Parallax --
BG_PARALLAX   = BG_PARALLAX   if 'BG_PARALLAX' in dir() else 2.5
BG_BLUR       = BG_BLUR       if 'BG_BLUR'     in dir() else 3
NUM_LAYERS    = NUM_LAYERS    if 'NUM_LAYERS'   in dir() else 10

# -- Compositing --
AO_WIDTH      = AO_WIDTH      if 'AO_WIDTH'    in dir() else 6
AO_STRENGTH   = AO_STRENGTH   if 'AO_STRENGTH' in dir() else 0.35

# -- Inpainting --
MASK_DILATE_K = MASK_DILATE_K if 'MASK_DILATE_K' in dir() else 21
MASK_DILATE_I = MASK_DILATE_I if 'MASK_DILATE_I' in dir() else 5

# -- Depth Model (Large for A100, override to Small-hf for weaker GPUs) --
DEPTH_MODEL   = DEPTH_MODEL   if 'DEPTH_MODEL' in dir() else 'depth-anything/Depth-Anything-V2-Large-hf'

OUTPUT = Path(INPUT).stem + '_3d.gif'
OUTPUT_MP4 = Path(INPUT).stem + '_3d.mp4'

print(f'Config: FRAMES={FRAMES} ARC_DEG={ARC_DEG} FPS={FPS} NUM_LAYERS={NUM_LAYERS} MAX_DIM={MAX_DIM}')
print(f'Depth model: {DEPTH_MODEL}')

## 3. Run Pipeline

In [ ]:
import cv2
import torch
import numpy as np
import time
import os
import warnings
warnings.filterwarnings('ignore')

from PIL import Image
from tqdm import tqdm
from IPython.display import display, Image as IPImage

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# ── Helper functions ──

def guided_filter(guide, target, radius=8, eps=0.01):
    """Edge-preserving smoothing using a guided filter."""
    guide = guide.astype(np.float64)
    target = target.astype(np.float64)
    ksize = 2 * radius + 1
    mean_g = cv2.boxFilter(guide, -1, (ksize, ksize))
    mean_t = cv2.boxFilter(target, -1, (ksize, ksize))
    mean_gt = cv2.boxFilter(guide * target, -1, (ksize, ksize))
    mean_gg = cv2.boxFilter(guide * guide, -1, (ksize, ksize))
    a = (mean_gt - mean_g * mean_t) / (mean_gg - mean_g * mean_g + eps)
    b = mean_t - a * mean_g
    mean_a = cv2.boxFilter(a, -1, (ksize, ksize))
    mean_b = cv2.boxFilter(b, -1, (ksize, ksize))
    return (mean_a * guide + mean_b).astype(np.float32)


def iterative_inpaint(img_bgr, m, passes=5):
    """Multi-pass OpenCV inpainting for large masked areas."""
    result = img_bgr.copy()
    remaining = m.copy()
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    for _ in range(passes):
        eroded = cv2.erode(remaining, k, iterations=3)
        band = np.clip(remaining.astype(np.float32) - eroded.astype(np.float32), 0, 1).astype(np.uint8)
        if band.sum() < 10:
            break
        result = cv2.inpaint(result, band, 7, cv2.INPAINT_NS)
        remaining = eroded
    if remaining.sum() > 0:
        result = cv2.inpaint(result, remaining, 15, cv2.INPAINT_TELEA)
    return result


def inpaint_lama(img_rgb, mask_binary):
    """LaMa inpainting with OpenCV fallback."""
    try:
        from simple_lama_inpainting import SimpleLama
        print('  Using LaMa inpainting...')
        lama = SimpleLama()
        img_pil = Image.fromarray(img_rgb)
        mask_pil = Image.fromarray((mask_binary * 255).astype(np.uint8))
        result_pil = lama(img_pil, mask_pil)
        return np.array(result_pil)
    except Exception as e:
        print(f'  LaMa failed ({e}), falling back to OpenCV...')
        img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        result_bgr = iterative_inpaint(img_bgr, mask_binary.astype(np.uint8))
        return cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB)


def build_ao_shadow(mask, width=6, strength=0.35):
    """Generate ambient occlusion shadow at mask edges."""
    dist = cv2.distanceTransform((mask > 0.5).astype(np.uint8), cv2.DIST_L2, 5)
    dist = np.clip(dist / max(width, 1), 0, 1)
    ao = 1.0 - (1.0 - dist) * strength * (mask > 0.5).astype(np.float32)
    return cv2.GaussianBlur(np.clip(ao, 0, 1).astype(np.float32), (5, 5), 1.5)


def depth_adaptive_blur(image, depth_field, max_blur=5):
    """Apply continuous depth-of-field blur that scales with depth.
    Deeper regions get more blur, shallower regions stay sharper."""
    if max_blur <= 0:
        return image
    result = image.astype(np.float32)
    # Build 4 blur levels and blend based on depth
    blur_levels = [
        result.copy(),
        cv2.GaussianBlur(result, (0, 0), max_blur * 0.5),
        cv2.GaussianBlur(result, (0, 0), max_blur),
        cv2.GaussianBlur(result, (0, 0), max_blur * 2.0),
    ]
    # depth_field: higher = farther = more blur
    d = np.clip(depth_field, 0, 1)
    # Power ramp for more natural DoF falloff
    d = d ** 1.5
    # Tri-linear blend across 4 levels
    idx = d * 3.0  # map 0-1 to 0-3
    idx = np.clip(idx, 0, 2.999)
    lo = np.floor(idx).astype(np.int32)
    frac = (idx - lo)[:, :, None]
    out = np.zeros_like(result)
    for i in range(3):
        mask_i = (lo == i)
        if mask_i.any():
            mask_3 = mask_i[:, :, None]
            out += mask_3 * (blur_levels[i] * (1.0 - frac) + blur_levels[i + 1] * frac)
    return np.clip(out, 0, 255).astype(np.uint8)


def sinusoidal_ease(t_linear):
    """Apply sinusoidal easing for more natural camera motion.
    Slow at extremes, faster through center — mimics real camera pans."""
    return np.sin(t_linear * np.pi / 2.0)

In [ ]:
# ── Load image ──

raw = Image.open(INPUT).convert('RGB')
w, h = raw.size
if max(w, h) > MAX_DIM:
    s = MAX_DIM / max(w, h)
    w, h = int(w * s) & ~1, int(h * s) & ~1
    raw = raw.resize((w, h), Image.LANCZOS)
img = np.array(raw)
H, W = img.shape[:2]
print(f'Image: {W}x{H}')

In [ ]:
# ── Stage 1: Depth Estimation (Large model + float16 + bilateral refinement) ──

t0 = time.time()
print(f'Stage 1: Depth Estimation ({DEPTH_MODEL.split("/")[-1]})')

from transformers import AutoImageProcessor, AutoModelForDepthEstimation

depth_proc = AutoImageProcessor.from_pretrained(DEPTH_MODEL)

# Use float16 on CUDA for faster inference with no quality loss
_dtype = torch.float16 if device == 'cuda' else torch.float32
depth_model = AutoModelForDepthEstimation.from_pretrained(
    DEPTH_MODEL, torch_dtype=_dtype
).to(device).eval()

inputs = depth_proc(images=Image.fromarray(img), return_tensors='pt').to(device)
if device == 'cuda':
    inputs['pixel_values'] = inputs['pixel_values'].to(_dtype)

with torch.no_grad():
    pred = depth_model(**inputs).predicted_depth

depth_raw = torch.nn.functional.interpolate(
    pred.float().unsqueeze(1), size=(H, W), mode='bicubic', align_corners=False
)[0, 0].cpu().numpy().astype(np.float32)
depth = (depth_raw - depth_raw.min()) / (depth_raw.max() - depth_raw.min() + 1e-8)

del depth_model, depth_proc
torch.cuda.empty_cache()

# Auto-detect polarity (near=0, far=1)
center_d = depth[H//3:2*H//3, W//3:2*W//3].mean()
edge_d = np.concatenate([depth[0,:], depth[-1,:], depth[:,0], depth[:,-1]]).mean()
if center_d > edge_d:
    depth = 1.0 - depth
    print('  (inverted depth polarity)')

# Refine depth edges — guided filter preserves edges from the RGB image
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
depth = guided_filter(gray, depth, radius=8, eps=0.02)
depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

# Bilateral filter pass — smooths flat depth regions while preserving
# the sharp depth boundaries the guided filter locked in.
# This reduces depth noise that causes jittery parallax in flat areas.
depth_u8 = (depth * 255).astype(np.uint8)
depth_bilateral = cv2.bilateralFilter(depth_u8, d=9, sigmaColor=25, sigmaSpace=25)
depth = depth_bilateral.astype(np.float32) / 255.0
depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

print(f'  Done in {time.time()-t0:.1f}s')
cv2.imwrite('depth.png', cv2.applyColorMap((depth * 255).astype(np.uint8), cv2.COLORMAP_INFERNO))

In [ ]:
# ── Stage 2: Subject Segmentation (float16 + edge-aware alpha matting) ──

t0 = time.time()
print('Stage 2: Segmentation')

from transformers import AutoModelForImageSegmentation
from torchvision.transforms.functional import normalize

_dtype = torch.float16 if device == 'cuda' else torch.float32
rmbg = AutoModelForImageSegmentation.from_pretrained(
    'briaai/RMBG-2.0', trust_remote_code=True,
    torch_dtype=_dtype
).to(device).eval()

t_img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
t_img = normalize(t_img, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
t_img = torch.nn.functional.interpolate(
    t_img.unsqueeze(0), (1024, 1024), mode='bilinear', align_corners=False
).to(device)
if device == 'cuda':
    t_img = t_img.to(_dtype)

with torch.no_grad():
    mask_raw = torch.sigmoid(rmbg(t_img)[-1][0, 0]).float().cpu().numpy()

# Soft alpha — preserve the model's continuous 0-to-1 edge detail
mask_soft = cv2.resize(mask_raw, (W, H)).astype(np.float32)
kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
mask_soft = cv2.morphologyEx(mask_soft, cv2.MORPH_CLOSE, kern, iterations=3)

# Edge-aware alpha matting — refine mask edges using the image as guide.
# This locks the alpha boundary to actual color edges (hair, fur, etc.)
# producing cleaner composites with less fringing.
mask_soft = guided_filter(
    gray,
    mask_soft,
    radius=12,
    eps=0.005
)
mask_soft = np.clip(mask_soft, 0, 1)

# Binary mask for inpainting and layer assignment
mask_binary = (mask_soft > 0.5).astype(np.uint8)
mask_binary = cv2.morphologyEx(mask_binary, cv2.MORPH_OPEN, kern, iterations=1)

del rmbg
torch.cuda.empty_cache()

print(f'  Foreground: {mask_binary.mean()*100:.1f}%  Done in {time.time()-t0:.1f}s')
cv2.imwrite('mask.png', (mask_soft * 255).astype(np.uint8))

In [ ]:
# ── Stage 3: Multi-Layer Inpainting (with depth-adaptive DoF blur) ──

t0 = time.time()
print('Stage 3: Multi-Layer Inpainting')

# --- Quantize depth into layers using percentile thresholds ---
bg_pixels = depth[mask_binary == 0]
if len(bg_pixels) > 0:
    pcts = np.linspace(0, 100, NUM_LAYERS + 1)[1:-1]
    thresholds = np.percentile(bg_pixels, pcts)
else:
    thresholds = np.linspace(0, 1, NUM_LAYERS + 1)[1:-1]

print(f'  Depth thresholds: {thresholds}')

# Build per-layer binary masks (layer 0 = nearest, layer N-1 = farthest)
layer_masks = []
for i in range(NUM_LAYERS):
    if i == 0:
        m = depth <= thresholds[0]
    elif i == NUM_LAYERS - 1:
        m = depth > thresholds[-1]
    else:
        m = (depth > thresholds[i - 1]) & (depth <= thresholds[i])
    layer_masks.append(m.astype(np.float32))

# Force subject into the nearest layer (layer 0)
for i in range(1, NUM_LAYERS):
    layer_masks[i][mask_binary > 0] = 0.0
layer_masks[0][mask_binary > 0] = 1.0

# Clean layer masks
kern_clean = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
for i in range(NUM_LAYERS):
    lm = layer_masks[i].astype(np.uint8)
    lm = cv2.morphologyEx(lm, cv2.MORPH_CLOSE, kern_clean, iterations=2)
    lm = cv2.morphologyEx(lm, cv2.MORPH_OPEN, kern_clean, iterations=1)
    layer_masks[i] = lm.astype(np.float32)

# Merge tiny layers (<2% of pixels) into nearest neighbor
for i in range(NUM_LAYERS):
    if layer_masks[i].mean() < 0.02 and i != 0:
        merge_target = i - 1 if i > 0 else i + 1
        if merge_target < NUM_LAYERS:
            layer_masks[merge_target] = np.clip(
                layer_masks[merge_target] + layer_masks[i], 0, 1
            )
            layer_masks[i] = np.zeros_like(layer_masks[i])
            print(f'  Layer {i} merged into layer {merge_target} (<2% pixels)')

# Assign any unclaimed pixels to the farthest layer
claimed = np.zeros((H, W), dtype=np.float32)
for lm in layer_masks:
    claimed = np.clip(claimed + lm, 0, 1)
unclaimed = (claimed < 0.5).astype(np.float32)
if unclaimed.sum() > 0:
    for i in reversed(range(NUM_LAYERS)):
        if layer_masks[i].sum() > 0:
            layer_masks[i] = np.clip(layer_masks[i] + unclaimed, 0, 1)
            print(f'  Assigned {unclaimed.mean()*100:.1f}% unclaimed pixels to layer {i}')
            break

for i in range(NUM_LAYERS):
    pct = layer_masks[i].mean() * 100
    print(f'  Layer {i}: {pct:.1f}% of image')

# --- Build backing plates back-to-front ---
dilate_kern = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (MASK_DILATE_K, MASK_DILATE_K)
)

# Fully-inpainted background (subject removed)
full_inpaint_mask = cv2.dilate(
    mask_binary,
    dilate_kern,
    iterations=MASK_DILATE_I
)
bg_full = inpaint_lama(img, full_inpaint_mask)
if bg_full.shape[:2] != (H, W):
    bg_full = cv2.resize(bg_full, (W, H), interpolation=cv2.INTER_LANCZOS4)

layers = []

for i in reversed(range(NUM_LAYERS)):
    lm_binary = (layer_masks[i] > 0.5).astype(np.uint8)
    if lm_binary.sum() == 0:
        continue

    # Everything in front of this layer needs inpainting
    behind_mask = np.zeros((H, W), dtype=np.uint8)
    for j in range(i):
        behind_mask = np.clip(behind_mask + (layer_masks[j] > 0.5).astype(np.uint8), 0, 1)

    inpaint_region = cv2.dilate(behind_mask, dilate_kern, iterations=MASK_DILATE_I)

    if inpaint_region.sum() > 0:
        layer_image = inpaint_lama(img, inpaint_region)
        if layer_image.shape[:2] != (H, W):
            layer_image = cv2.resize(layer_image, (W, H), interpolation=cv2.INTER_LANCZOS4)
    else:
        layer_image = img.copy()

    # Depth-adaptive blur for non-subject layers
    if i != 0 and BG_BLUR > 0:
        layer_image = depth_adaptive_blur(layer_image, depth, max_blur=BG_BLUR)

    # Inpaint depth for this layer
    d_u8 = (depth * 255).astype(np.uint8)
    if inpaint_region.sum() > 0:
        d_u8_masked = d_u8.copy()
        d_u8_masked[inpaint_region > 0] = 0
        layer_depth = cv2.inpaint(d_u8_masked, inpaint_region, 15, cv2.INPAINT_NS).astype(np.float32) / 255.0
    else:
        layer_depth = depth.copy()

    # Alpha: soft for subject layer, binary for mid layers
    if i == 0:
        layer_alpha = mask_soft.copy()
    else:
        layer_alpha = layer_masks[i].copy()

    # Disparity multiplier
    if NUM_LAYERS > 1:
        disparity_mult = 1.0 + (BG_PARALLAX - 1.0) * (i / (NUM_LAYERS - 1))
    else:
        disparity_mult = 1.0

    layers.append({
        'image': layer_image,
        'depth': layer_depth,
        'alpha': layer_alpha,
        'disparity_mult': disparity_mult,
        'index': i,
    })

# Sort back-to-front (farthest first)
layers.sort(key=lambda l: -l['index'])

# Farthest layer = base plate, full alpha, full coverage
if layers:
    farthest = layers[0]
    farthest['alpha'] = np.ones((H, W), dtype=np.float32)
    if BG_BLUR > 0:
        farthest['image'] = depth_adaptive_blur(bg_full, depth, max_blur=BG_BLUR)
    else:
        farthest['image'] = bg_full.copy()

# Ambient occlusion
ao_map = build_ao_shadow(mask_binary.astype(np.float32), width=AO_WIDTH, strength=AO_STRENGTH)

print(f'  {len(layers)} active layers  Done in {time.time()-t0:.1f}s')
Image.fromarray(bg_full).save('bg_inpainted.png')

In [ ]:
# ── Stage 4: Orbital Rendering (Lanczos4 + sinusoidal easing) ──

t0 = time.time()
print('Stage 4: Orbital Rendering')

max_shift = W * np.tan(np.radians(ARC_DEG)) * 0.3
angles_linear = np.linspace(-1.0, 1.0, FRAMES)
# Sinusoidal easing — slow at extremes, faster through center
angles = np.array([sinusoidal_ease(t) for t in angles_linear])

# Subject centroid anchor — keeps subject stable in frame
fg_ys, fg_xs = np.where(mask_binary > 0)
if len(fg_xs) > 0:
    subject_center_disparity = (1.0 - np.median(depth[fg_ys, fg_xs])) * max_shift
else:
    subject_center_disparity = 0.0

# Static coordinate grids for cv2.remap
u_grid = np.arange(W, dtype=np.float32)[None, :].repeat(H, axis=0)
v_grid = np.arange(H, dtype=np.float32)[:, None].repeat(W, axis=1)

# Precompute per-layer disparity fields
for layer in layers:
    layer['disparity'] = (1.0 - layer['depth']) * max_shift * layer['disparity_mult']

frames = []
for t_val in tqdm(angles, desc='Rendering'):
    anchor_offset = subject_center_disparity * t_val
    result = np.zeros((H, W, 3), dtype=np.float32)

    # Composite back-to-front (farthest layer first)
    for layer in layers:
        # Inverse warp: for each destination pixel, look back at source
        shift_field = layer['disparity'] * t_val - anchor_offset
        map_x = (u_grid - shift_field).astype(np.float32)
        map_y = v_grid.astype(np.float32)

        # Warp image — Lanczos4 (8x8 kernel) for publication-quality sub-pixel shifts
        warped_img = cv2.remap(
            layer['image'].astype(np.float32), map_x, map_y,
            cv2.INTER_LANCZOS4, borderMode=cv2.BORDER_REFLECT
        )

        # Warp alpha — Lanczos4 for crisp soft-edge preservation
        warped_alpha = cv2.remap(
            layer['alpha'].astype(np.float32), map_x, map_y,
            cv2.INTER_LANCZOS4, borderMode=cv2.BORDER_CONSTANT
        )

        # Apply AO shadow to subject layer
        if layer['index'] == 0:
            warped_ao = cv2.remap(
                ao_map, map_x, map_y,
                cv2.INTER_LANCZOS4, borderMode=cv2.BORDER_CONSTANT, borderValue=1.0
            )
            for c in range(3):
                warped_img[:, :, c] *= warped_ao

        # Tighter alpha smoothing kernel (3x3 vs old 5x5) to preserve detail
        alpha = cv2.GaussianBlur(warped_alpha, (3, 3), 0.6)[:, :, None]
        alpha = np.clip(alpha, 0, 1)

        # Composite
        result = result * (1.0 - alpha) + warped_img * alpha

    frames.append(np.clip(result, 0, 255).astype(np.uint8))

print(f'  {len(frames)} frames  Done in {time.time()-t0:.1f}s')

In [ ]:
# ── Stage 5: GIF + MP4 Assembly ──

print('Stage 5: Assembly')

if PING_PONG and len(frames) > 2:
    gif_frames = frames + frames[-2:0:-1]
else:
    gif_frames = frames

# --- Save individual PNGs first (needed for both GIF and MP4) ---
os.makedirs('frames', exist_ok=True)
for i, f in enumerate(gif_frames):
    Image.fromarray(f).save(f'frames/frame_{i:03d}.png')
print(f'  Saved {len(gif_frames)} PNGs to frames/')

# --- High-quality GIF with optimized quantization ---
pil_frames = [Image.fromarray(f) for f in gif_frames]

# Quantize each frame with median-cut for better color distribution
quantized = []
for pf in pil_frames:
    q = pf.quantize(colors=256, method=Image.Quantize.MEDIANCUT, dither=Image.Dither.FLOYDSTEINBERG)
    quantized.append(q)

quantized[0].save(
    OUTPUT, save_all=True, append_images=quantized[1:],
    duration=int(1000 / FPS), loop=0, optimize=True
)

size_mb = os.path.getsize(OUTPUT) / 1024**2
print(f'  {OUTPUT}: {len(gif_frames)} frames, {size_mb:.1f} MB')

# --- MP4 output (high-quality H.264, no GIF color banding) ---
try:
    import subprocess
    mp4_cmd = [
        'ffmpeg', '-y',
        '-framerate', str(FPS),
        '-i', 'frames/frame_%03d.png',
        '-c:v', 'libx264',
        '-preset', 'slow',
        '-crf', '18',
        '-pix_fmt', 'yuv420p',
        '-vf', 'pad=ceil(iw/2)*2:ceil(ih/2)*2',
        OUTPUT_MP4
    ]
    subprocess.run(mp4_cmd, capture_output=True, check=True)
    mp4_mb = os.path.getsize(OUTPUT_MP4) / 1024**2
    print(f'  {OUTPUT_MP4}: {mp4_mb:.1f} MB (no color banding, shareable)')
except Exception as e:
    print(f'  MP4 encoding skipped ({e})')

display(IPImage(filename=OUTPUT))

## 4. Results & Download

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (title, path) in zip(axes, [
    ('Input', INPUT), ('Depth', 'depth.png'),
    ('Mask', 'mask.png'), ('BG Inpainted', 'bg_inpainted.png')
]):
    ax.imshow(Image.open(path))
    ax.set_title(title, fontsize=14)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import files
    files.download(OUTPUT)
    if os.path.exists(OUTPUT_MP4):
        files.download(OUTPUT_MP4)

    # Optional: download frames zip for lenticular interlacing
    # import shutil
    # shutil.make_archive('lenticular_frames', 'zip', 'frames')
    # files.download('lenticular_frames.zip')
else:
    print(f'Output saved to: {OUTPUT}')
    if os.path.exists(OUTPUT_MP4):
        print(f'MP4 saved to: {OUTPUT_MP4}')